# CodeGen — Group 45
## Step 4: RAG — retrieval-augmented Python→Rust translation (Checkpoint 3)

**What we are building:** the RAG layer on top of our best model (the Step 3b **translation**
model, 7.1%). At answer time we take the query's Python, retrieve the **top-K most similar
validated Python→Rust pairs** from our own training set, and prepend them to the prompt as
worked examples. We then sweep **K ∈ {0, 1, 2, 4}** on HumanEval-Rust with the same Step-1
harness. **K=0 is the control** — it rebuilds the exact Step 3b prompt, so it must reproduce
~7.1%, and any change at K≥1 is attributable to retrieval alone.

**Design choices (sized for free Colab T4):**
- **TF-IDF retrieval** (scikit-learn, char n-grams): CPU-only, deterministic, nothing to
  download. A code-embedding retriever (CodeBERT) is a stretch upgrade — the `retrieve()`
  function is the only thing to swap.
- **Crash-safe sweep:** every result streams to `rag_eval_k{K}.jsonl` as it happens;
  re-running skips finished problems (same pattern as the Step 2 engine). A disconnect
  costs nothing, and different K's can even run on different Colab accounts in parallel.
- **Smoke test first** (Section 7): ~2 minutes end-to-end before we commit GPU hours.
- **Free error analysis:** the harness now also captures `rustc` stderr, so at the end we
  get a table of the Rust error codes that block us — the error-analysis section of the report.

**Leakage:** the RAG index contains **only MBPP-derived training pairs**; we evaluate on
HumanEval-Rust. Retrieved examples are data the model already saw in training — legitimate
context, never test answers.

**To run:** T4 GPU runtime, Drive with the Step 2.5 base + Step 3b adapter + pairs file,
then `Runtime → Run all`.

## 1. Install Rust + libraries
Same rules as before: do **not** upgrade `torch`; add `peft` + `scikit-learn`; drop old `torchao`.

In [ ]:
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version
# IMPORTANT: do NOT add `torch` here (Colab's torch/torchvision must stay matched).
!pip install -q -U datasets transformers accelerate peft scikit-learn
!pip uninstall -q -y torchao
print("setup done")

## 2. Mount Drive, load the pairs + check artifacts
We reuse three Drive artifacts: the **Rust-aware base** (Step 2.5), the **translation adapter**
(Step 3b), and the **validated pairs** (Step 2) — the pairs double as the RAG corpus.

In [ ]:
import os, json, shutil
from google.colab import drive
drive.mount("/content/drive")

DRIVE   = "/content/drive/MyDrive/CodeGen_Group45"
BASE    = DRIVE + "/codegen350m-rust-base"            # Rust-aware base (Step 2.5)
ADAPTER = DRIVE + "/codegen350m-rust-lora-translate"  # translation LoRA (Step 3b)

# The training pairs double as the RAG corpus (prefer the newest set).
PAIRS = None
for cand in ["pairs_v3.jsonl", "pairs_v2.jsonl"]:
    if os.path.exists(DRIVE + "/" + cand):
        PAIRS = cand
        if not os.path.exists(PAIRS):
            shutil.copy(DRIVE + "/" + cand, PAIRS)
        break
assert PAIRS, "no pairs_v*.jsonl found in " + DRIVE
assert os.path.isdir(BASE),    "Rust-aware base not found: " + BASE
assert os.path.isdir(ADAPTER), "translation adapter not found: " + ADAPTER

with open(PAIRS) as f:
    pairs = [json.loads(l) for l in f]
print(len(pairs), "pairs from", PAIRS)
print("base   :", BASE)
print("adapter:", ADAPTER)

## 3. Load the translation model (base + LoRA adapter)
Exactly the Step 3b eval setup — same base, same adapter, fp32, greedy decoding later —
so the K=0 control is a true re-measurement of the 7.1% model, not a near-copy.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE)     # fp32, like the Step 3b eval
model = PeftModel.from_pretrained(model, ADAPTER)      # attach the translation LoRA
model.config.pad_token_id = tok.eos_token_id
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print("translation model loaded on", model.device)

## 4. Build the RAG index (TF-IDF over the Python side)
The corpus is **one exemplar per training task** — the shortest passing Rust solution, because
concise examples cost fewer context tokens. We index the **Python** source with character
n-gram TF-IDF (robust for code, ignores identifier spelling quirks) and retrieve by cosine
similarity: *query Python in → most similar validated Python→Rust pairs out*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# One exemplar per task: the SHORTEST passing solution (saves context tokens).
best = {}
for p in pairs:
    t = p["task"]
    if t not in best or len(p["rust_solution"]) < len(best[t]["rust_solution"]):
        best[t] = p
corpus = list(best.values())
print(len(corpus), "unique tasks in the RAG index (from", len(pairs), "pairs)")

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=50000)
X = vec.fit_transform([c["python"] for c in corpus])

def retrieve(query_python, k):
    if k <= 0:
        return []
    sims = cosine_similarity(vec.transform([query_python]), X)[0]
    order = sims.argsort()[::-1][:k]
    return [corpus[i] for i in order]

# Sanity check: querying with one of our own tasks should retrieve itself first.
demo = retrieve(corpus[0]["python"], 3)
print("query task :", corpus[0]["task"])
print("retrieved  :", [d["task"] for d in demo])
assert demo[0]["task"] == corpus[0]["task"], "retriever failed its own-task sanity check"
print("retriever OK")

## 5. The RAG prompt (with a hard token budget)
Each retrieved pair is rendered **exactly like a training example** (Python as a comment, then
the full Rust solution), stacked in front of the query. Critical detail: codegen-350M's context
is **2048 tokens** and we generate up to 512, so the prompt gets a hard budget of 1536 tokens —
examples that would overflow are skipped (nearest-first), and we record how many actually fit.
With K=0 the prompt is **byte-identical to Step 3b's** — that's our control.

In [ ]:
CTX, MAX_NEW = 2048, 512          # codegen-350M context window; generation headroom
PROMPT_BUDGET = CTX - MAX_NEW

def python_as_comment(py):
    if not py:
        return ""
    body = "\n".join("// " + line for line in py.strip().splitlines())
    return "// Reference Python implementation:\n" + body + "\n"

def ntokens(s):
    return len(tok(s)["input_ids"])

def build_rag_prompt(query_python, rust_prompt, k):
    query = python_as_comment(query_python) + rust_prompt   # == the Step 3b prompt
    blocks, total = [], ntokens(query)
    for ex in retrieve(query_python, k):                    # nearest first
        block = python_as_comment(ex["python"]) + ex["rust_solution"] + "\n\n"
        bt = ntokens(block)
        if total + bt > PROMPT_BUDGET:
            continue                                        # would overflow the context — skip
        blocks.append(block); total += bt
    return "".join(blocks) + query, len(blocks)

p, n = build_rag_prompt(corpus[5]["python"], corpus[5]["rust_prompt"], 2)
print("demo prompt:", n, "retrieved examples,", ntokens(p), "tokens")
print("-" * 60)
print(p[:1500])

## 6. Harness + eval set (HumanEval-Rust with its Python)
Same `evaluate_one` verdicts as Steps 1–3b, with one addition: we keep `rustc`'s **stderr** for
every failure, which powers the error analysis in Section 10. Same fixed `trim_to_body`.

In [ ]:
import re, subprocess, tempfile
from collections import Counter
from datasets import load_dataset

def evaluate_one_detailed(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    # Same verdicts as the Step-1 harness, but also returns stderr for error analysis.
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src, binp = os.path.join(wd, "main.rs"), os.path.join(wd, "prog")
        open(src, "w").write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout", ""
        if c.returncode != 0:
            return "compile_error", c.stderr[:800]
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout", ""
        return ("pass", "") if r.returncode == 0 else ("run_fail", r.stderr[:400])

def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text

def load_rs(cfg):
    try:    return load_dataset("nuprl/MultiPL-E", cfg, split="test")
    except Exception: return load_dataset("nuprl/MultiPL-E", cfg, split="test", trust_remote_code=True)

eval_ds = load_rs("humaneval-rs")

pyset = load_dataset("openai/openai_humaneval", split="test")
py_by_id = {int(ex["task_id"].split("/")[1]): ex["prompt"] + ex["canonical_solution"]
            for ex in pyset}

def rs_id(name):
    m = re.search(r"HumanEval_(\d+)_", name)
    return int(m.group(1)) if m else None

print(len(eval_ds), "eval problems;",
      sum(rs_id(ex["name"]) in py_by_id for ex in eval_ds), "matched to Python")

## 7. The crash-safe sweep runner (+ smoke test)
`run_k(k)` evaluates one K value over the benchmark, **streaming each result to
`rag_eval_k{k}.jsonl` immediately**. Re-running skips finished problems, so a disconnect
loses at most one problem. Records keep the body, the example count and the stderr —
everything the analysis cells need.

The smoke test below runs **5 problems at K=2** (~2 min) to prove the whole path
(retrieve → prompt → generate → compile) before we commit GPU hours. It writes to a
separate `_smoke` file and deletes it, so real results are never touched.

In [ ]:
def model_completion(prompt_text):
    inputs = tok(prompt_text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

def run_k(k, limit=None, tag=""):
    path = f"rag_eval_k{k}{tag}.jsonl"
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            done = {json.loads(l)["name"] for l in f}
    data = eval_ds if limit is None else eval_ds.select(range(limit))
    todo = [ex for ex in data if ex["name"] not in done]
    print(f"K={k}: {len(done)} already done, {len(todo)} to go -> {path}")
    with open(path, "a") as f:
        for i, ex in enumerate(todo, 1):
            py = py_by_id.get(rs_id(ex["name"]), "")
            prompt_text, n_used = build_rag_prompt(py, ex["prompt"], k)
            body = model_completion(prompt_text)
            status, err = evaluate_one_detailed(ex["prompt"], body, ex["tests"])
            f.write(json.dumps({"name": ex["name"], "k": k, "n_examples": n_used,
                                "status": status, "err": err, "body": body}) + "\n")
            f.flush()
            if i % 10 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)}")
    with open(path) as f:
        statuses = [json.loads(l)["status"] for l in f]
    counts = Counter(statuses)
    print(f"K={k}: pass {100*counts['pass']/len(statuses):.1f}%  {dict(counts)}")
    return counts

# --- smoke test: 5 problems at K=2, throwaway file ---
run_k(2, limit=5, tag="_smoke")
os.remove("rag_eval_k2_smoke.jsonl")
print("\nsmoke test OK — the full sweep below is safe to launch")

## 8. The full K sweep (the Checkpoint 3 experiment)
**K=0 runs first — it is the control** and should land at ~7.1% (Step 3b). If it doesn't,
stop and check the Drive paths in Section 2 before burning GPU time on the other K's.

Budget: each K takes roughly 20–35 min on a T4 (~1.5–2 h total). The K's are **independent
files**, so this also splits across Colab accounts: run `[0, 1]` on one account and `[2, 4]`
on another (same Drive artifacts), then put all `rag_eval_k*.jsonl` in the Drive folder.
If a session dies mid-run, just `Run all` again — finished problems are skipped.

In [ ]:
for k in [0, 1, 2, 4]:
    run_k(k)

## 9. Results table — the RAG ablation
Pass rate per K next to the fixed reference numbers. `avg examples` shows how many retrieved
examples actually fit the context budget (if it is well below K, the budget — not retrieval —
is the binding constraint).

In [ ]:
import glob
print("Reference: vanilla 1.3% | fine-tuned generation 0.6% | translation (Step 3b) 7.1% = the K=0 row\n")
rows = []
for path in sorted(glob.glob("rag_eval_k*.jsonl")):
    if "_smoke" in path:
        continue
    with open(path) as f:
        recs = [json.loads(l) for l in f]
    if not recs:
        continue
    k = recs[0]["k"]
    c = Counter(r["status"] for r in recs)
    avg_ex = sum(r["n_examples"] for r in recs) / len(recs)
    rows.append((k, len(recs), 100 * c["pass"] / len(recs), c["compile_error"],
                 c["run_fail"], c["run_timeout"] + c["compile_timeout"], avg_ex))
rows.sort()
print(f"{'K':>3} {'n':>4} {'pass%':>7} {'compile_err':>12} {'run_fail':>9} {'timeout':>8} {'avg examples':>13}")
for k, n, p, ce, rf, to, ae in rows:
    print(f"{k:>3} {n:>4} {p:>6.1f}% {ce:>12} {rf:>9} {to:>8} {ae:>13.2f}")

## 10. Error analysis — what actually blocks us
We count the `rustc` error codes across every compile failure of the K=0 run (the plain
translation model). This tells us *which Rust concepts the model gets wrong* — type
mismatches? borrows? unknown methods? — and therefore what training data to generate more of.
This table goes straight into the report.

In [ ]:
with open("rag_eval_k0.jsonl") as f:
    recs = [json.loads(l) for l in f]

codes = Counter()
for r in recs:
    if r["status"] == "compile_error":
        for c in set(re.findall(r"error\[(E\d+)\]", r["err"])):
            codes[c] += 1

print("top rustc error codes (number of problems affected):")
for c, n in codes.most_common(10):
    print(f"  {c}: {n:>3}   (see: rustc --explain {c})")

fail = next((r for r in recs if r["status"] == "compile_error"), None)
if fail:
    print("\n=== one failure end-to-end:", fail["name"], "===")
    print(fail["err"][:700])

## 10b. Flip analysis + compile-guided cascade (our best result)
Section 9's straight-line result is negative, but RAG is **not uniformly harmful** — each K
solves a few problems K=0 cannot (mostly math/number-theory, where our MBPP index is dense),
while losing others (simple list-processing the model already solved unaided). The union of
passes across K ∈ {0,1,2,4} is the **oracle ceiling** (measured: 19/156 = 12.2%).

The cascade turns part of that ceiling into a **deployable policy** using only signals legal
at inference time — the *compiler*, never the tests: take the K=0 output; if it fails to
compile, fall back to the K=1 output, then K=4. Measured: **7.1% → 10.3%** (16/156).

*Honesty notes for the report:* (1) the fallback rule is parameter-free, but the cascade
order was chosen after seeing the sweep — say so; the `[1,4]` and `[1,2,4]` cascades score
identically (10.3%), so the choice isn't load-bearing. (2) Cost: a fallback generation +
compile per stage, only for outputs that fail to compile.

In [ ]:
def load_k(k):
    with open(f"rag_eval_k{k}.jsonl") as f:
        return {r["name"]: r for r in map(json.loads, f)}

base = load_k(0)
for k in [1, 2, 4]:
    cur = load_k(k)
    gained = [n for n in cur if cur[n]["status"] == "pass" and base[n]["status"] != "pass"]
    lost   = [n for n in cur if cur[n]["status"] != "pass" and base[n]["status"] == "pass"]
    print(f"K={k}:  +{len(gained)} newly pass: {gained}")
    print(f"       -{len(lost)} lost:       {lost}\n")

oracle = {n for n, r in base.items() if r["status"] == "pass"}
for k in [1, 2, 4]:
    oracle |= {n for n, r in load_k(k).items() if r["status"] == "pass"}
print(f"oracle ceiling (union of passes over K): {len(oracle)}/{len(base)}"
      f" = {100*len(oracle)/len(base):.1f}%")

In [ ]:
for ks in [[1], [2], [4], [1, 4], [1, 2, 4]]:
    alts = [load_k(k) for k in ks]
    total = 0
    for n, r0 in base.items():
        if r0["status"] == "pass":
            total += 1                     # K=0 output compiles & passes -> kept
        elif r0["status"].startswith("compile"):
            # K=0 didn't compile -> fall back down the cascade
            for cur in alts:
                if cur[n]["status"] == "pass":
                    total += 1
                    break
                if not cur[n]["status"].startswith("compile"):
                    break                  # this one compiles (but is wrong) -> policy keeps it
    print(f"cascade 0 -> {ks}: {100*total/len(base):.1f}%")

## 11. Save everything to Drive

In [ ]:
for path in glob.glob("rag_eval_k*.jsonl"):
    if "_smoke" not in path:
        shutil.copy(path, DRIVE + "/" + path)
print("results copied to", DRIVE)

## What we built (and what's next)
- The **RAG layer** of the proposal: TF-IDF retrieval of validated Python→Rust pairs,
  prepended as in-context examples, with a **top-K ablation** on HumanEval-Rust — the core
  Checkpoint 3 deliverable.
- A **controlled comparison**: K=0 reproduces the Step 3b translation model, so every delta
  in the Section 9 table is attributable to retrieval.
- **The compile-guided cascade** (Section 10b): 7.1% -> **10.3%** using only the compiler
  at inference time; oracle ceiling across K = 12.2%.
- **Error analysis for free**: the rustc error-code table (Section 10) — evidence for the
  report and a guide for what training data to generate next.

**Next:**
1. **Large-LLM baseline** through this same harness (API model, same 156 problems) — the last
   column of the Checkpoint 3 comparison table.
2. **Consolidated results table**: vanilla → +mono → +pairs (generation) → translation → +RAG → LLM.
3. **Wire artifacts into the repo demo**: adapter → `models/`, pairs → `data/`, then
   `RUSTGEN_BACKEND=hf` for a real end-to-end demo (Checkpoint 4).
4. Stretch: swap `retrieve()` for CodeBERT embeddings and re-run only the best K.